# Fase 1: Preprocesamiento y Extracción de Características

Este notebook se encarga de cargar el dataset original (transcripciones), realizar la limpieza de texto y extraer las 163 características originales (incluyendo análisis fraseológico, polaridad, subjetividad, legibilidad, entropía, etc.).

In [ ]:
# 1. Conexión con Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Instalación de dependencias requeridas
!pip install emoji textstat textblob spacy nltk transformers imbalanced-learn xgboost

In [ ]:
# 3. Descarga de modelos lingüísticos y recursos NLTK
!python -m spacy download es_core_news_sm
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('vader_lexicon')

In [ ]:
# 4. Carga de librerías
import pandas as pd
import numpy as np
import re
import emoji
import textstat
import math
import unicodedata
from collections import Counter
from nltk.tokenize import sent_tokenize, word_tokenize
from textblob import TextBlob
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
import spacy
from transformers import pipeline
import torch
from tqdm.notebook import tqdm
tqdm.pandas()

### Definición de Funciones de Extracción de Características

In [ ]:
nlp = spacy.load("es_core_news_sm")
sia = SentimentIntensityAnalyzer()
stopwords_es = set(stopwords.words('spanish'))

# Inicializar detector de ironía preentrenado (Roberta)
irony_detector = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-irony",
    device=0 if torch.cuda.is_available() else -1
)

def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)
    text = emoji.replace_emoji(text, replace="")
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r"[^a-zA-Záéíóúüñ¿?¡!.,;]", " ", text)

    doc = nlp(text)
    text = " ".join([token.lemma_ for token in doc if token.text not in stopwords_es])
    return text

In [ ]:
# Cargar palabras comunes (CREA) y palabras satíricas
# NOTA: Ajusta estas rutas a donde tengas los archivos en tu Drive o entorno de Colab
CREA_PATH = "/content/drive/MyDrive/Titulacion/CREA_PalabrasComunes.txt"
SATIRE_WORDS_PATH = "/content/drive/MyDrive/Titulacion/adverbios_conectores_satira_expandido.csv"

df_CREA = pd.read_csv(CREA_PATH, sep='\t', encoding='latin-1')
common_words_es = set(df_CREA['Palabra'].dropna().tolist())

satirical_df = pd.read_csv(SATIRE_WORDS_PATH, sep=';')
satirical_words = satirical_df['PALABRAS'].dropna().tolist()

In [ ]:
def satira(text):
    doc = nlp(text)
    text_lower = text.lower()
    text_norm = ''.join(c for c in unicodedata.normalize('NFD', text_lower) if unicodedata.category(c) != 'Mn')
    text_clean = ''.join(c for c in text_norm if not c in '"#$%&\'()*+-/:<=>@[\\]^_`{|}~')

    satire_words_count = {}
    total_satire_words = 0
    for phrase in satirical_words:
        count = text_clean.count(phrase.lower())
        satire_words_count[phrase] = count
        total_satire_words += count

    word_count = len([token.text for token in doc if not token.is_punct])
    satire_words_density = total_satire_words / word_count if word_count else 0

    return satire_words_count, total_satire_words, satire_words_density

In [ ]:
def extract_all_features(text, processed_text):
    doc = nlp(processed_text)
    words_processed = [token.text.lower() for token in doc]
    num_words_proc = len(words_processed)

    # Básicos
    num_words = len(word_tokenize(text))
    num_chars = len(text)
    exclamations = text.count("!")
    uppercase_ratio = sum(1 for c in text if c.isupper()) / max(1, len(text))

    # Sentimiento e ironía
    blob = TextBlob(text)
    polarity = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity
    vader_polarity = sia.polarity_scores(text)['compound']
    
    try:
        irony_res = irony_detector(text[:512])[0]
        irony_score = irony_res['score'] if irony_res['label'] == 'irony' else 1 - irony_res['score']
    except:
        irony_score = 0.5

    # POS
    prop_ADV = sum(1 for token in doc if token.pos_ == "ADV") / num_words_proc if num_words_proc > 0 else 0
    prop_NOUN = sum(1 for token in doc if token.pos_ == "NOUN") / num_words_proc if num_words_proc > 0 else 0
    prop_VERB = sum(1 for token in doc if token.pos_ == "VERB") / num_words_proc if num_words_proc > 0 else 0
    prop_ADJ = sum(1 for token in doc if token.pos_ == "ADJ") / num_words_proc if num_words_proc > 0 else 0
    rhetorical_questions = sum(1 for sent in doc.sents if sent.text.strip().endswith("?"))
    metaphors = len(re.findall(r'\bcomo\b|\bes como\b', text, re.IGNORECASE))

    # Métricas fraseológicas individuales
    sat_counts, total_sat, sat_density = satira(processed_text)

    # Dependencias
    total_depth = 0
    total_length = 0
    sentence_count = 0
    for sent in doc.sents:
        depths = [token.head.i - token.i if token.head != token else 0 for token in sent]
        depth = max(depths) if depths else 0
        length = sum(abs(dep) for dep in depths)
        total_depth += depth
        total_length += length
        sentence_count += 1
    avg_depth = total_depth / sentence_count if sentence_count else 0
    avg_length = total_length / sentence_count if sentence_count else 0

    # Flesch
    sentences = [s.strip() for s in re.split(r'[.!?]+', processed_text) if s.strip()]
    num_sentences = len(sentences) if sentences else 1
    num_syllables = sum(textstat.syllable_count(word) for word in processed_text.split())
    asl = num_words_proc / num_sentences
    asw = num_syllables / max(1, num_words_proc)
    flesch = 206.84 - (1.02 * asl) - (60 * asw)

    # Entropía
    words_split = processed_text.lower().split()
    word_counts = Counter(words_split)
    total_words_split = sum(word_counts.values())
    entropy = -sum((count / total_words_split) * math.log2(count / total_words_split) for count in word_counts.values()) if total_words_split else 0

    # Repetición sintáctica
    dep_patterns = [token.dep_ for token in doc]
    dep_counts = Counter(dep_patterns)
    repetition_index = max(dep_counts.values()) / len(dep_patterns) if dep_patterns else 0

    # Frecuencia inusual
    words_in_text = set(word.lower() for word in processed_text.split())
    unusual_words = words_in_text - common_set if 'common_set' in locals() or 'common_words_es' in globals() else set()
    unusual_freq = len(unusual_words) / len(words_in_text) if words_in_text else 0

    # Diversidad léxica
    words_lex = re.findall(r"\w+", processed_text.lower())
    lexical_div = (len(set(words_lex)) / len(words_lex)) * 100 if words_lex else 0
    
    # Longitudes promedio
    words_clean = re.sub(r'[^\w\s]', '', processed_text.lower()).split()
    mean_word_len = np.mean([len(word) for word in words_clean]) if words_clean else 0
    mean_sent_len = np.mean([len(sent.split()) for sent in sentences]) if sentences else 0
    stdev_sent_len = np.std([len(sent.split()) for sent in sentences]) if len(sentences) >= 2 else 0
    doc_len = sum(len(s) for s in sentences)

    # Retornar todas las características mapeadas
    feats = {
        'MeanWordLen': mean_word_len,
        'LexicalDiversity': lexical_div,
        'MeanSentenceLen': mean_sent_len,
        'StdevSentenceLen': stdev_sent_len,
        'MeanParagraphLen': len(processed_text.split('\n')),
        'DocumentLen': doc_len,
        'WordsPerText': num_words_proc,
        'SentencesPerText': num_sentences,
        'MeanDifferenceSentenceLengths': 0.0,
        'num_words': num_words,
        'num_chars': num_chars,
        'exclamations': exclamations,
        'uppercase_ratio': uppercase_ratio,
        'polarity': polarity,
        'subjectivity': subjectivity,
        'Polaridad_VADER': vader_polarity,
        'irony_score': irony_score,
        'prop_ADV': prop_ADV,
        'prop_NOUN': prop_NOUN,
        'prop_VERB': prop_VERB,
        'prop_ADJ': prop_ADJ,
        'rhetorical_questions': rhetorical_questions,
        'metaphors': metaphors,
        'satire_words_density': sat_density,
        'total_satire_words': total_sat,
        'avg_depth': avg_depth,
        'avg_length': avg_length,
        'Flesch Score': flesch,
        'Lexical Entropy': entropy,
        'Syntactic Repetition': repetition_index,
        'Unusual Word Frequency': unusual_freq
    }
    # Mezclar conteos individuales de palabras satíricas
    feats.update(sat_counts)
    return feats

### Carga del dataset original y ejecución

In [ ]:
# Cargar tu dataset original. Reemplaza por la ruta de tu archivo (ej. un JSON o CSV con 'transcription' y 'label')
DATASET_PATH = "/content/drive/MyDrive/Titulacion/DatasetsFinales/dataset_base.csv"
df = pd.read_csv(DATASET_PATH)

# 1. Preprocesar texto
print("Preprocesando texto...")
df['transcription_processed'] = df['transcription'].progress_apply(preprocess_text)

# 2. Extraer características
print("Extrayendo características...")
feature_rows = []
for idx, row in tqdm(df.iterrows(), total=len(df)):
    feats = extract_all_features(row['transcription'], row['transcription_processed'])
    feats['id'] = row.get('id', f"doc_{idx}")
    feats['label'] = row['label']
    feats['transcription'] = row['transcription']
    feats['transcription_processed'] = row['transcription_processed']
    feature_rows.append(feats)

df_feat = pd.DataFrame(feature_rows)

# Guardar resultado
OUTPUT_PATH = "/content/drive/MyDrive/Titulacion/DatasetsFinales/df_train2_feat.jsonl"
df_feat.to_json(OUTPUT_PATH, orient='records', lines=True)
print(f"¡Completado! Dataset con características guardado en: {OUTPUT_PATH}")